In [1]:
# --- Setup Environment for FaceNet ---
!pip uninstall -y onnxruntime onnxruntime-gpu
!pip install -q facenet-pytorch chromadb scikit-learn psutil pandas matplotlib tqdm opencv-python-headless

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 138.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 97.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/2

In [2]:
import os
import sys
import time
import re
import subprocess
import warnings
import psutil
import logging
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image

import torch
import cv2
import chromadb
from google.colab import drive

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# OFFICIAL FACENET IMPLEMENTATION (MTCNN + INCEPTIONRESNETV1)
from facenet_pytorch import MTCNN, InceptionResnetV1

warnings.filterwarnings("ignore")

In [3]:
drive.mount("/content/drive", force_remount=False)

Mounted at /content/drive


In [4]:
# --- Logging Configuration ---
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - [%(levelname)s] - %(message)s'
)
logger = logging.getLogger(__name__)

# --- Paths & Constants ---
PROJECT_PATH = "/content/drive/MyDrive/face_recognition"
ENROLLMENT_PATH = os.path.join(PROJECT_PATH, "dataset", "enrollment")
TESTING_PATH = os.path.join(PROJECT_PATH, "dataset", "testing")
DATABASE_PATH = os.path.join(PROJECT_PATH, "chroma_db", "facenet_db")
COLLECTION_NAME = "facenet_faces"
MODEL_NAME = "FaceNet"
TOP_K = 1

BENCHMARK_DIR = os.path.join(PROJECT_PATH, "benchmark_results", "facenet")
os.makedirs(BENCHMARK_DIR, exist_ok=True)

# Set to True to wipe and re-index ChromaDB on every run
FORCE_REENROLL = True

# --- Initialize Hardware Monitoring & FaceNet Models ---
logger.info("Initializing Official FaceNet (MTCNN + InceptionResnetV1) Pipeline...")
start_model_load = time.perf_counter()

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [5]:
mtcnn = MTCNN(
    image_size=160,
    margin=0,
    min_face_size=20,
    thresholds=[0.6, 0.7, 0.7],
    factor=0.709,
    post_process=True,
    keep_all=True,
    device=device
)

# Feature Extractor (FaceNet InceptionResnetV1)
recognition_model = InceptionResnetV1(pretrained='vggface2').eval().to(device)

if torch.cuda.is_available():
    torch.cuda.synchronize()

MODEL_LOADING_TIME = time.perf_counter() - start_model_load

  0%|          | 0.00/107M [00:00<?, ?B/s]

In [8]:
def read_image_rgb(image_path):
    image_bgr = cv2.imread(image_path)
    if image_bgr is None:
        raise ValueError(f"Cannot read image: {image_path}")
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    return Image.fromarray(image_rgb)

def get_person_name(image_name):
    base = os.path.splitext(image_name)[0].lower()
    return re.sub(r'\d+$', '', base).strip('_')

def get_expected_identities(image_name):
    base = os.path.splitext(image_name)[0].lower()

    if "dr_strange" in base and "loki" in base:
        return ["dr_strange", "loki"]
    if "tony" in base and "natasha" in base:
        return ["tony", "natasha"]
    if "tony" in base and "steve" in base:
        return ["tony", "steve_roger"]

    name = re.sub(r'\d+$', '', base).strip('_')
    if "unknown" in name:
        return ["Unknown"]
    return [name]

def get_cpu_usage():
    return psutil.cpu_percent(interval=None)

def get_ram_usage():
    return psutil.virtual_memory().percent

def get_gpu_metrics():
    util = 0.0
    vram_mb = 0.0
    try:
        util_raw = subprocess.getoutput("nvidia-smi --query-gpu=utilization.gpu --format=csv,nounits,noheader")
        vram_raw = subprocess.getoutput("nvidia-smi --query-gpu=memory.used --format=csv,nounits,noheader")

        util_match = re.findall(r"\d+", util_raw)
        vram_match = re.findall(r"\d+", vram_raw)

        util = float(util_match[-1]) if util_match else 0.0
        vram_mb = float(vram_match[-1]) if vram_match else 0.0
    except (ValueError, IndexError):
        pass

    return util, vram_mb

def get_model_size_mb():
    # Calculate parameter and buffer memory for MTCNN + InceptionResnetV1
    total_bytes = sum(p.nelement() * p.element_size() for p in recognition_model.parameters())
    total_bytes += sum(b.nelement() * b.element_size() for b in recognition_model.buffers())
    return total_bytes / (1024 * 1024)

logger.info(f"Model Loaded in: {MODEL_LOADING_TIME:.4f} sec (Size: {get_model_size_mb():.2f} MB)")

In [9]:
# Initialize Vector DB with L2 Distance
client = chromadb.PersistentClient(path=DATABASE_PATH)

if FORCE_REENROLL:
    try:
        client.delete_collection(COLLECTION_NAME)
    except Exception:
        pass

collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "l2"} # Official L2 metric space for FaceNet
)

In [10]:
def extract_facenet_embeddings(pil_image):
    """
    Executes MTCNN detection, alignment, and FaceNet InceptionResnetV1 forward pass.
    """
    # MTCNN detects, crops, aligns, and standardizes tensor to (N, 3, 160, 160)
    faces_tensor = mtcnn(pil_image)
    if faces_tensor is None:
        return []

    with torch.no_grad():
        faces_tensor = faces_tensor.to(device)
        embeddings = recognition_model(faces_tensor)

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        embeddings_np = embeddings.cpu().numpy()

    return embeddings_np

In [12]:
def enroll_images():
    logger.info("--- Starting FaceNet Enrollment ---")
    image_files = sorted(os.listdir(ENROLLMENT_PATH))

    for image_name in tqdm(image_files, desc="Enrolling"):
        image_path = os.path.join(ENROLLMENT_PATH, image_name)
        try:
            pil_img = read_image_rgb(image_path)
        except ValueError:
            continue

        embeddings = extract_facenet_embeddings(pil_img)
        if len(embeddings) != 1:
            logger.warning(f"Skipped {image_name}: Found {len(embeddings)} faces (enrollment requires exactly 1 face).")
            continue

        embedding = embeddings[0].tolist()
        person_name = get_person_name(image_name)

        collection.upsert(
            ids=[image_name],
            embeddings=[embedding],
            metadatas=[{
                "person_name": person_name,
                "image_name": image_name,
                "image_path": image_path,
                "model": MODEL_NAME
            }]
        )

logger.info(f"Enrollment Complete! Total enrolled identities: {collection.count()}")

In [13]:
# --- Query & Search Pipeline ---
def search_image(image_path):
    try:
        pil_img = read_image_rgb(image_path)
    except ValueError as e:
        logger.warning(f"Cannot read image {image_path}: {e}")
        return []

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0_feat = time.perf_counter()

    embeddings = extract_facenet_embeddings(pil_img)
    if len(embeddings) == 0:
        return []

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    feature_extraction_time = time.perf_counter() - t0_feat

    predictions = []
    total_search_time = 0.0

    for query_embedding in embeddings:
        t0_search = time.perf_counter()
        results = collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=TOP_K,
            include=["metadatas", "distances"]
        )
        search_time = time.perf_counter() - t0_search
        total_search_time += search_time

        if not results["distances"] or not results["distances"][0]:
            best_match = "Unknown"
            l2_dist = float('inf')
        else:
            # ChromaDB L2 space computes squared Euclidean distance (d^2)
            # We take square root to get true L2 Distance d = sqrt(sum((a-b)^2))
            squared_l2 = results["distances"][0][0]
            l2_dist = np.sqrt(max(0.0, squared_l2))
            best_match = results["metadatas"][0][0]["person_name"].replace(" ", "_")

        predictions.append({
            "Best_Match": best_match,
            "L2_Distance": l2_dist,
            "Feature_Extraction_Time": feature_extraction_time,
            "Search_Time": search_time,
        })

    total_inference = feature_extraction_time + total_search_time
    for p in predictions:
        p["Inference_Time"] = total_inference

    return predictions

In [14]:
# --- Automated Evaluation & Threshold Grid Search ---
def evaluate_system():
    if collection.count() == 0:
        logger.error("Database is empty. Evaluation aborted.")
        return

    logger.info("--- Starting FaceNet Testing & Performance Evaluation ---")

    test_files = sorted(os.listdir(TESTING_PATH))

    all_searches = []
    cpu_records = []
    ram_records = []
    gpu_util_records = []
    global_peak_vram = 0.0

    get_cpu_usage() # Warmup CPU reading

    # 1. Execute raw inference across testing dataset
    for image_name in test_files:
        image_path = os.path.join(TESTING_PATH, image_name)
        expected_identities = [name.replace(" ", "_") for name in get_expected_identities(image_name)]
        preds = search_image(image_path)

        all_searches.append({
            "image_name": image_name,
            "preds": preds,
            "expected": expected_identities
        })

        cpu_records.append(get_cpu_usage())
        ram_records.append(get_ram_usage())
        g_util, g_vram = get_gpu_metrics()
        gpu_util_records.append(g_util)

        if g_vram > global_peak_vram:
            global_peak_vram = g_vram

    # 2. Automated 1D Threshold Search for L2 Distance (Maximizing Weighted F1-Score)
    # Standard FaceNet optimal L2 distance threshold ranges typically between 0.6 and 1.2
    L2_THRESHOLDS = np.arange(0.3, 1.5, 0.025)

    best_f1 = -1.0
    best_l2_threshold = 2.0
    best_raw_predictions = []
    best_y_true = []
    best_y_pred = []

    logger.info("Running automated threshold optimization for FaceNet L2 Distance...")

    for l2_thresh in L2_THRESHOLDS:
        temp_raw = []
        temp_y_true = []
        temp_y_pred = []

        for item in all_searches:
            expected = item["expected"].copy()
            preds = item["preds"]
            frame_preds = []

            for p in preds:
                p_copy = p.copy()
                p_copy["Image_Name"] = item["image_name"]

                is_match = p_copy["Best_Match"] in expected
                is_close = p_copy["L2_Distance"] <= l2_thresh

                if is_match and is_close:
                    p_copy["Ground_Truth"] = p_copy["Best_Match"]
                    expected.remove(p_copy["Best_Match"])
                else:
                    p_copy["Ground_Truth"] = None
                frame_preds.append(p_copy)

            for p_copy in frame_preds:
                if p_copy["Ground_Truth"] is None:
                    if expected:
                        p_copy["Ground_Truth"] = expected.pop(0)
                    else:
                        p_copy["Ground_Truth"] = "Spurious Detection"

            temp_raw.extend(frame_preds)

            for missing in expected:
                temp_raw.append({
                    "Image_Name": item["image_name"],
                    "Ground_Truth": missing,
                    "Best_Match": "Missed",
                    "L2_Distance": float('inf'),
                    "Feature_Extraction_Time": 0.0,
                    "Search_Time": 0.0,
                    "Inference_Time": 0.0
                })

        for p in temp_raw:
            if p["Best_Match"] == "Missed":
                temp_y_pred.append("Unknown (Missed)")
                temp_y_true.append(p["Ground_Truth"])
            else:
                final_pred = p["Best_Match"] if p["L2_Distance"] <= l2_thresh else "Unknown"
                temp_y_pred.append(final_pred)
                temp_y_true.append(p["Ground_Truth"])

        current_f1 = f1_score(temp_y_true, temp_y_pred, average="weighted", zero_division=0)

        if current_f1 > best_f1:
            best_f1 = current_f1
            best_l2_threshold = l2_thresh
            best_raw_predictions = temp_raw
            best_y_true = temp_y_true
            best_y_pred = temp_y_pred

    logger.info(f"Optimal Threshold Selected -> L2 Distance: {best_l2_threshold:.3f} (F1: {best_f1*100:.2f}%)")

    # 3. Compile predictions and metrics for CSV export
    final_csv_data = []

    for item in best_raw_predictions:
        if item["Best_Match"] == "Missed":
            final_pred = "Unknown (Missed)"
            status = "Failed"
            correct_flag = "Incorrect"
        else:
            final_pred = item["Best_Match"] if item["L2_Distance"] <= best_l2_threshold else "Unknown"
            status = "Known" if final_pred != "Unknown" else "Unknown"
            correct_flag = "Correct" if final_pred == item["Ground_Truth"] else "Incorrect"

        final_csv_data.append({
            "Image_Name": item["Image_Name"],
            "Ground_Truth": item["Ground_Truth"],
            "Prediction": final_pred,
            "Status": status,
            "L2_Distance": round(item["L2_Distance"], 4) if item["L2_Distance"] != float('inf') else "N/A",
            "Correct/Incorrect": correct_flag,
            "Feature_Extraction_Time": item["Feature_Extraction_Time"],
            "Search_Time": item["Search_Time"],
            "Inference_Time": item["Inference_Time"]
        })

    df_preds = pd.DataFrame(final_csv_data)
    df_preds.to_csv(os.path.join(BENCHMARK_DIR, f"{MODEL_NAME.lower()}_predictions.csv"), index=False)

    accuracy = accuracy_score(best_y_true, best_y_pred)
    precision = precision_score(best_y_true, best_y_pred, average="weighted", zero_division=0)
    recall = recall_score(best_y_true, best_y_pred, average="weighted", zero_division=0)

    labels = sorted(list(set(best_y_true) | set(best_y_pred)))
    cm = confusion_matrix(best_y_true, best_y_pred, labels=labels)
    df_cm = pd.DataFrame(cm, index=labels, columns=labels)
    df_cm.to_csv(os.path.join(BENCHMARK_DIR, f"{MODEL_NAME.lower()}_confusion_matrix.csv"))

    avg_feat_time = df_preds[df_preds["Feature_Extraction_Time"] > 0]["Feature_Extraction_Time"].mean()
    avg_search_time = df_preds[df_preds["Search_Time"] > 0]["Search_Time"].mean()
    avg_inference_time = df_preds[df_preds["Inference_Time"] > 0]["Inference_Time"].mean()

    avg_cpu = sum(cpu_records) / len(cpu_records) if cpu_records else 0.0
    avg_ram = sum(ram_records) / len(ram_records) if ram_records else 0.0
    avg_gpu_util = sum(gpu_util_records) / len(gpu_util_records) if gpu_util_records else 0.0

    if global_peak_vram > 0 or avg_gpu_util > 0:
        gpu_usage_str = f"Util: {avg_gpu_util:.1f}%, Peak VRAM: {global_peak_vram:.0f} MB"
    else:
        gpu_usage_str = "No GPU / CUDA Idle"

    results_data = [{
        "Model": MODEL_NAME,
        "Selected_L2_Distance_Threshold": round(best_l2_threshold, 3),
        "Accuracy_Percent": round(accuracy * 100, 2),
        "Precision_Percent": round(precision * 100, 2),
        "Recall_Percent": round(recall * 100, 2),
        "F1_Score_Percent": round(best_f1 * 100, 2),
        "Average_Feature_Extraction_Time_sec": round(avg_feat_time if not pd.isna(avg_feat_time) else 0, 6),
        "Average_Search_Time_sec": round(avg_search_time if not pd.isna(avg_search_time) else 0, 6),
        "Average_Inference_Time_sec": round(avg_inference_time if not pd.isna(avg_inference_time) else 0, 6),
        "Model_Loading_Time_sec": round(MODEL_LOADING_TIME, 6),
        "Embedding_Dimension": 512,
        "Model_Size_MB": round(get_model_size_mb(), 2),
        "CPU_Usage_Percent": round(avg_cpu, 2),
        "RAM_Usage_Percent": round(avg_ram, 2),
        "GPU_VRAM_Usage": gpu_usage_str
    }]

    df_results = pd.DataFrame(results_data)
    df_results.to_csv(os.path.join(BENCHMARK_DIR, f"{MODEL_NAME.lower()}_results.csv"), index=False)

    logger.info("\n" + "="*60)
    logger.info("         FACENET BENCHMARK PERFORMANCE REPORT")
    logger.info("="*60)
    logger.info(f"Total Faces Evaluated : {len(best_y_true)}")
    logger.info(f"Optimum L2 Threshold  : {best_l2_threshold:.3f}")
    logger.info(f"System Accuracy       : {accuracy * 100:.2f}%")
    logger.info(f"System Precision      : {precision * 100:.2f}%")
    logger.info(f"System Recall         : {recall * 100:.2f}%")
    logger.info(f"System F1-Score       : {best_f1 * 100:.2f}%")
    logger.info("-" * 60)
    logger.info(f"Avg Inference Time    : {round(avg_inference_time if not pd.isna(avg_inference_time) else 0, 4)} sec")
    logger.info(f"GPU Monitor Status    : {gpu_usage_str}")
    logger.info("="*60)
    logger.info(f"Data Exported to: {BENCHMARK_DIR}")

# --- Main Driver ---
if __name__ == "__main__":
    if FORCE_REENROLL or collection.count() == 0:
        enroll_images()

    evaluate_system()

Enrolling: 100%|██████████| 7/7 [00:09<00:00,  1.37s/it]
